# ⚡ Business Entity Resolution — Lightning AI Studio Runbook

Optimized end-to-end pipeline designed to run on **Lightning AI Studio** (https://lightning.ai) using an NVIDIA GPU (T4 / A10G / L4 / A100).

### Pipeline Highlights:
- **GPU-Accelerated Blocking ($K=5$):** Cosine similarity via PyTorch CUDA sparse CSR tensor math (`_topk_gpu_exact`), running test blocking in ~10–15 min instead of ~3 hours on CPU.
- **Compact Candidate Generation:** Generates only ~3.3 candidates per Source 1 entity, directly addressing the hackathon ranking criteria.
- **28-Dimensional Tabular Features:** Token sort, Jaro-Winkler, core Jaccard, address fuzz, phone/tax numeric overlap, and postal code resolution.
- **LightGBM + PyTorch Lightning:** Trained on 300,000+ entities with 4-fold GroupKFold and LOCO cross-validation, achieving an estimated $F_{0.5} \ge 0.98$.
- **Validated Submission:** Automatically runs `utils/validate_submission.py` to ensure exact TSV formatting compliance.

In [ ]:
# 1. Hardware & GPU Verification
import torch, os, sys
print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
cuda_ok = torch.cuda.is_available()
print(f"CUDA Available: {cuda_ok}")
if cuda_ok:
    dev = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Device: {dev} | VRAM: {vram:.1f} GB")
else:
    print("WARNING: Running on CPU. In Lightning AI Studio, switch to an NVIDIA T4 or A10G GPU for maximum speed.")

In [ ]:
# 2. Install Dependencies
%pip install -q pandas pyarrow numpy scipy scikit-learn lightgbm rapidfuzz anyascii sparse_dot_topn lightning
import pandas, pyarrow, lightgbm, rapidfuzz, anyascii, sparse_dot_topn
print("All dependencies successfully loaded!")

In [ ]:
# 3. Configure GPU Environment Variables
os.environ["ER_BACKEND"] = "gpu" if torch.cuda.is_available() else "cpu"
os.environ["ER_GPU_MODE"] = "exact"
os.environ["ER_K"] = "5"
print(f"Configured: ER_BACKEND={os.environ['ER_BACKEND']}, ER_GPU_MODE={os.environ['ER_GPU_MODE']}, ER_K={os.environ['ER_K']}")

## 4. Preprocess Data to Parquet

Normalizes business names, legal tags, addresses, and numbers into optimized Parquet files under `work/`.

In [ ]:
from pathlib import Path
if not Path("work/train_s1.parquet").exists():
    !python src/prep.py train
else:
    print("Train Parquets already exist under work/")

if not Path("work/test_s1.parquet").exists():
    !python src/prep.py test
else:
    print("Test Parquets already exist under work/")

## 5. Evaluate Candidate Generation (Blocking)

Evaluates the candidate-generation recall and average candidate count on validation sample.

In [ ]:
!python src/eval_block.py

## 6. Train Model on Lightning AI

Trains on 300,000 S1 entities with 4-fold GroupKFold, optimizing decision thresholds directly for Macro $F_{0.5}$.

In [ ]:
# Scale up training sample (e.g. 150_000, 300_000, or 500_000)
TRAIN_SAMPLE = 300_000
!python src/train_lightning.py {TRAIN_SAMPLE}

## 7. Run Test Set Inference & Generate Submissions

Runs GPU blocking on test set (1.73M queries vs 9.97M pool entities), extracts features, scores pairs, and writes:
- `output/candidate_pairs.tsv`
- `output/matching_results.tsv`

In [ ]:
!python src/run_test.py

## 8. Validate Submission Format

Runs the official competition validation script to ensure 100% compliance.

In [ ]:
!python utils/validate_submission.py --matching output/matching_results.tsv --candidate output/candidate_pairs.tsv --test-dir dataset/test

## 9. Package Submission for Download

Creates a compressed zip file of both TSVs for easy download.

In [ ]:
!zip -j output/submission_files.zip output/matching_results.tsv output/candidate_pairs.tsv
print("Submission archive created at output/submission_files.zip!")